<a href="https://colab.research.google.com/github/Loopinlogix/Market_Analysis_Project-2/blob/main/Stock_Market_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stock Market Analysis Project 2


## Intro to this Project

This notebook is all about digging into some historical stock market data. We're gonna do a bunch of things: grab the data, clean it up (get rid of weird errors and bad values), find any crazy outliers, check for duplicates, cook up some new features from the existing data, make sure everything's on the same scale, and then split it all up so we can eventually build some machine learning models.

Basically, we've got two main data files: one with general info about stocks (`historical_stocks.csv`) like where they're traded, their names, what industry they're in, etc., and another with the daily prices and trading volumes (`historical_stock_prices.csv`).

The whole point here is to take all that raw, messy stock info and turn it into something neat and organized, packed with useful features. This way, we'll have a solid dataset ready to go for training models to try and figure out what the stock market might do next.

In [ ]:

#Github

#Github
!apt-get install -y git
!git config --global user.email "crystal_macneil@hotmail.com"
!git config --global user.name "Crystal MacNeil"

!git clone https://github.com/Loopinlogix/Market_Analysis_Project-2.git
%cd Market_Analysis_Project-2
!ls


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'Market_Analysis_Project-2'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Market_Analysis_Project-2
README.md


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("=" * 60)
print("STEP 1: DATA COLLECTION")
print("=" * 60)

stocks = pd.read_csv('historical_stocks.csv')
prices = pd.read_csv('historical_stock_prices.csv')

# STANDARDIZE COLUMN NAMES TO LOWERCASE
prices.columns = prices.columns.str.strip().str.lower()
stocks.columns = stocks.columns.str.strip().str.lower()

print(prices.head())
print(stocks.head())
print(prices.info())

print("=" * 60)
print("STEP 2: ADVANCED DATA CLEANING")
print("=" * 60)

# Convert Date Column and sort
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values('date')
prices = prices.set_index('date')

print("Missing values before advanced cleaning:\n", prices.isnull().sum())

# Forward/backward fill for price columns
prices[['open', 'high', 'low', 'close']] = prices[['open', 'high', 'low', 'close']].ffill().bfill()

# Interpolate volume
prices['volume'] = prices['volume'].interpolate(method='linear')

# Basic diagnostics
print("\n--- DIAGNOSTICS AFTER FILLING ---")
print("Unique values per column:")
print(prices[['open', 'high', 'low', 'close', 'volume']].nunique())
print("\nStandard deviation:")
print(prices[['open', 'high', 'low', 'close', 'volume']].std())
print("\nFirst 5 rows:")
print(prices[['open', 'high', 'low', 'close', 'volume']].head())

print("=" * 60)
print("STEP 3: OUTLIER DETECTION & CAPPING")
print("=" * 60)

def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return np.clip(series, lower, upper)

prices['close'] = cap_outliers(prices['close'])
prices['volume'] = cap_outliers(prices['volume'])

print("Outliers capped for 'close' and 'volume'.")

print("=" * 60)
print("STEP 4: ERROR IDENTIFICATION & CORRECTION")
print("=" * 60)

# Replace impossible values with NaN
for col in ['open', 'high', 'low', 'close']:
    prices.loc[prices[col] <= 0, col] = np.nan

prices.loc[prices['volume'] < 0, 'volume'] = np.nan

# Fix cases where high < low
mask = prices['high'] < prices['low']
prices.loc[mask, ['high', 'low']] = prices.loc[mask, ['low', 'high']].values

# Re-interpolate after corrections
prices = prices.interpolate()

print("Errors corrected and data re-interpolated.")
print("Remaining missing values:\n", prices.isnull().sum())

print("=" * 60)
print("STEP 5: MERGE DATASETS")
print("=" * 60)

print("Prices columns:", prices.columns.tolist())
print("Stocks columns:", stocks.columns.tolist())

merged = pd.merge(prices.reset_index(), stocks, on='ticker')
merged = merged.set_index('date')
print(merged.head())

print("=" * 60)
print("STEP 6: FEATURE ENGINEERING")
print("=" * 60)

# Rolling averages
merged['ma_7'] = merged['close'].rolling(7).mean()
merged['ma_30'] = merged['close'].rolling(30).mean()

# Volatility (rolling std)
merged['volatility_7'] = merged['close'].rolling(7).std()

# Daily returns
merged['returns'] = merged['close'].pct_change()

# RSI (Relative Strength Index)
delta = merged['close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
merged['rsi_14'] = 100 - (100 / (1 + rs))

print("Engineered features added: ma_7, ma_30, volatility_7, returns, rsi_14.")
print(merged[['close', 'ma_7', 'ma_30', 'volatility_7', 'returns', 'rsi_14']].head())

print("=" * 60)
print("STEP 7: DATA NORMALIZATION / STANDARDIZATION")
print("=" * 60)

scaler = StandardScaler()
num_cols = ['open', 'high', 'low', 'close', 'volume',
            'returns', 'ma_7', 'ma_30', 'volatility_7', 'rsi_14']

merged[num_cols] = scaler.fit_transform(merged[num_cols])

print("Numerical columns standardized.")
print(merged[num_cols].head())

print("=" * 60)
print("STEP 8: ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

cat_cols = ['ticker', 'sector', 'industry']
merged = pd.get_dummies(merged, columns=cat_cols, drop_first=True)

print("Categorical variables encoded with one-hot encoding.")
print("Columns after encoding:", merged.columns.tolist()[:30], "...")

print("=" * 60)
print("STEP 9: FINAL CLEANUP & CONSOLIDATION")
print("=" * 60)

final_data = merged.dropna()
print("Final dataset shape after dropping remaining NaNs:", final_data.shape)

print("=" * 60)
print("STEP 10: DATA SPLITTING FOR MODELING")
print("=" * 60)

train, temp = train_test_split(final_data, test_size=0.3, shuffle=False)
val, test = train_test_split(temp, test_size=0.5, shuffle=False)

print("Train set shape:", train.shape)
print("Validation set shape:", val.shape)
print("Test set shape:", test.shape)

print("=" * 60)
print("STEP 11: SAVE CLEANED DATASETS")
print("=" * 60)

train.to_csv('clean_train.csv')
val.to_csv('clean_val.csv')
test.to_csv('clean_test.csv')
final_data.to_csv('clean_full_dataset.csv')

print("Cleaned datasets saved as:")
print(" - clean_train.csv")
print(" - clean_val.csv")
print(" - clean_test.csv")
print(" - clean_full_dataset.csv")



STEP 1: DATA COLLECTION
  ticker   open  close  adj_close    low   high     volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900.0  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013-05-14
  ticker exchange                                    name             sector  \
0    PIH   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
1  PIHPP   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
2   TURN   NASDAQ                180 DEGREE CAPITAL CORP.            FINANCE   
3   FLWS   NASDAQ                 1-800 FLOWERS.COM, INC.  CONSUMER SERVICES   
4   FCCY   NASDAQ           1ST CONSTITUTION BANCORP (NJ)            FINANCE   

                     industry  
0  PROPERTY-CASUALTY INSURERS  
1  PR

/tmp/ipykernel_880/3048962541.py:81: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  prices = prices.interpolate()


Remaining missing values:
 ticker       0
open         0
close        0
adj_close    0
low          0
high         0
volume       0
dtype: int64
STEP 5: MERGE DATASETS
Prices columns: ['ticker', 'open', 'close', 'adj_close', 'low', 'high', 'volume']
Stocks columns: ['ticker', 'exchange', 'name', 'sector', 'industry']
           ticker      open     close  adj_close       low      high  \
date                                                                   
1970-01-02    MMM  6.851562  6.851562   0.438697  6.843750  6.890625   
1970-01-05    MMM  6.859375  6.890625   0.441198  6.859375  6.898438   
1970-01-06    MMM  6.890625  6.960938   0.445700  6.882812  6.960938   
1970-01-07    MMM  6.960938  7.000000   0.448201  6.945312  7.015625   
1970-01-08    MMM  7.000000  7.093750   0.454204  6.984375  7.109375   

              volume exchange        name       sector  \
date                                                     
1970-01-02   72000.0     NYSE  3M COMPANY  HEALTH CARE   
19